# Job Search with LLM Enhancement
Uses LLM to optimize search queries and filter relevant jobs

In [ ]:
#%pip install requests beautifulsoup4 transformers torch
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json

# Load LLM
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("✓ LLM loaded successfully")

## Step 1: Define Your Profile
Update these based on what you're looking for

In [ ]:
# Your profile (update as we go)
USER_PROFILE = {
    'target_roles': ['Software Engineer', 'Python Developer', 'Backend Engineer'],
    'experience_level': 'Junior to Mid-level',
    'skills': ['Python', 'JavaScript', 'SQL', 'Flask', 'React'],
    'preferred_companies': ['Startup', 'Tech Company', 'Remote-first'],
    'location': 'Remote',
    'salary_expectation': '$80,000 - $120,000',
    'must_haves': ['Remote', 'Growth opportunity'],
    'nice_to_haves': ['Equity', 'Flexible hours', 'Health insurance']
}

print("Your Profile:")
print(json.dumps(USER_PROFILE, indent=2))

## Step 2: LLM-Enhanced Search Query Generation
Use LLM to create better search terms

In [ ]:
def enhance_search_query(user_profile):
    """Use LLM to generate optimized search queries"""
    
    prompt = f"""You are a job search expert. Based on this profile, generate 3 optimized job search queries:
    
    Target Roles: {', '.join(user_profile['target_roles'])}
    Skills: {', '.join(user_profile['skills'])}
    Location: {user_profile['location']}
    Experience: {user_profile['experience_level']}
    
    Generate search queries that will find the most relevant jobs. Return only the queries, one per line:
    """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=300,
        temperature=0.7,
        num_return_sequences=1
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the new queries (after the prompt)
    queries = response.split(prompt)[-1].strip().split('\n')
    queries = [q.strip() for q in queries if q.strip() and len(q.strip()) > 5]
    
    return queries[:3]

# Generate search queries
search_queries = enhance_search_query(USER_PROFILE)
print("Generated Search Queries:")
for i, query in enumerate(search_queries, 1):
    print(f"{i}. {query}")

## Step 3: Search Jobs from Google
Scrape jobs based on optimized queries

In [ ]:
def search_jobs_google(query, num_results=5):
    """Search for jobs via Google"""
    # Using Indeed since it appears in Google results and is easier to scrape
    search_query = f"{query} site:indeed.com"
    url = f"https://www.google.com/search?q={search_query}"
    
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        jobs = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'indeed.com' in href and '/jobs?' in href:
                jobs.append({
                    'title': link.get_text(),
                    'url': href,
                    'source': 'Indeed'
                })
                if len(jobs) >= num_results:
                    break
        
        return jobs
    except Exception as e:
        print(f"Error searching: {e}")
        return []

# Search using all queries
all_jobs = []
for query in search_queries:
    print(f"\nSearching: {query}")
    jobs = search_jobs_google(query, num_results=3)
    all_jobs.extend(jobs)
    print(f"Found {len(jobs)} jobs")

print(f"\nTotal jobs found: {len(all_jobs)}")

## Step 4: LLM-Based Job Relevance Filtering
Use LLM to evaluate job relevance to your profile

In [ ]:
def evaluate_job_relevance(job, user_profile):
    """Use LLM to score job relevance"""
    
    prompt = f"""Rate this job's relevance (0-10) for someone with this profile:
    
    PROFILE:
    - Target Roles: {', '.join(user_profile['target_roles'])}
    - Skills: {', '.join(user_profile['skills'])}
    - Must Haves: {', '.join(user_profile['must_haves'])}
    - Location: {user_profile['location']}
    
    JOB:
    - Title: {job['title']}
    - Source: {job['source']}
    
    Score (0-10) and brief reason:
    """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=200,
        temperature=0.5
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Simple scoring: look for number at start
    try:
        score = int(response.split()[-1]) if any(c.isdigit() for c in response) else 5
        score = min(10, max(0, score))
    except:
        score = 5
    
    return score

print("Evaluating job relevance...\n")
job_scores = []

for job in all_jobs[:5]:  # Limit to first 5 for speed
    score = evaluate_job_relevance(job, USER_PROFILE)
    job_scores.append({
        'title': job['title'],
        'url': job['url'],
        'score': score,
        'relevant': score >= 6
    })
    print(f"[{score}/10] {job['title']}")

print(f"\nFound {sum(1 for j in job_scores if j['relevant'])} relevant jobs")

## Step 5: Display Relevant Jobs
Show filtered and ranked results

In [ ]:
# Sort by relevance score
relevant_jobs = sorted([j for j in job_scores if j['relevant']], key=lambda x: x['score'], reverse=True)

print("=" * 60)
print("RELEVANT JOBS FOR YOU")
print("=" * 60)

for i, job in enumerate(relevant_jobs, 1):
    print(f"\n{i}. [{job['score']}/10] {job['title']}")
    print(f"   Link: {job['url']}")

if not relevant_jobs:
    print("\nNo highly relevant jobs found. Adjust filters below and try again.")

## Step 6: Feedback & Iteration
Update your profile based on results

In [ ]:
# FEEDBACK: Update based on what you found
    # If no good results, try adjusting:

# Option 1: Broaden target roles
# USER_PROFILE['target_roles'].append('Full Stack Developer')

# Option 2: Expand skills
# USER_PROFILE['skills'].extend(['AWS', 'Docker'])

# Option 3: Relax location requirement
# USER_PROFILE['location'] = 'Remote or USA'

# Option 4: Lower salary expectation
# USER_PROFILE['salary_expectation'] = '$70,000 - $100,000'

# Option 5: Remove non-critical "must haves"
# USER_PROFILE['must_haves'] = ['Growth opportunity']

print("Tips to improve results:")
print("1. Comment on Options 1-5 above to adjust your profile")
print("2. Re-run all cells to get new results")
print("3. Keep iterating until you find roles you like")
print("\nOnce you find a good job, use the URL to apply!")